# 🚀 Sheikh-Max: Interleaved Thinking Coder

This notebook fine-tunes **Qwen 2.5 Coder 7B** to exhibit "Interleaved Thinking" - reasoning inside `<think>` tags before generating code.

**Optimized for Google Colab T4 GPU (15GB VRAM)**

## What this notebook does:
1. ✅ Install required libraries (Unsloth, TRL, PEFT)
2. ✅ Load the base Qwen 2.5 Coder model with 4-bit quantization
3. ✅ Prepare dataset with interleaved thinking format
4. ✅ Fine-tune with QLoRA on T4 GPU
5. ✅ Test the model for `<think>` tags
6. ✅ Push to Hugging Face Hub

---
## 1️⃣ Install Required Libraries

In [ ]:
%%capture
# Install Unsloth for optimized training (2x faster, 60% less memory)
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets wandb huggingface_hub

: 

---
## 2️⃣ Setup Environment

In [ ]:
import os
import gc
import torch

# Enable expandable memory segments for PyTorch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU available! This notebook requires a GPU.")
    print("   Go to Runtime > Change runtime type > GPU")

---
## 3️⃣ Authentication (Optional)

Set up Hugging Face and Weights & Biases for model pushing and experiment tracking.

In [ ]:
# Option 1: Use Colab secrets (recommended)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
    print("✅ Loaded tokens from Colab secrets")
except:
    # Option 2: Set manually
    HF_TOKEN = None  # Set your token here or leave None
    WANDB_API_KEY = None
    print("⚠️ No Colab secrets found. Set tokens manually if needed.")

# Set environment variables
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("✅ HF_TOKEN set")

if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print("✅ W&B logged in")

---
## 4️⃣ Load Base Model

Load Qwen 2.5 Coder 7B with 4-bit quantization using Unsloth.

In [ ]:
from unsloth import FastLanguageModel

# Model configuration
MODEL_ID = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048  # Can increase for longer contexts

print(f"📥 Loading model: {MODEL_ID}")
print(f"📏 Max sequence length: {MAX_SEQ_LENGTH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,  # T4 doesn't support bf16
    load_in_4bit=True,    # 4-bit quantization for memory efficiency
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Model loaded successfully!")
print(f"   Vocab size: {len(tokenizer)}")

---
## 5️⃣ Define Sheikh-Max Chat Template

This template formats conversations with `<think>` tags for interleaved thinking.

In [ ]:
# Sheikh-Max Chat Template with <think> tag support
SHEIKH_CHAT_TEMPLATE = """{{- bos_token }}
{%- for message in messages %}
    {%- if message['role'] == 'system' %}
        {%- if message['content'] %}
{{- '### System:\n' + message['content'].strip() + '\n\n' }}
        {%- endif %}
    {%- elif message['role'] == 'user' %}
{{- '### Instruction:\n' + message['content'].strip() + '\n\n' }}
    {%- elif message['role'] == 'assistant' %}
{{- '### Response:\n' }}
        {%- if message.get('thinking') %}
{{- '<think>\n' + message['thinking'].strip() + '\n</think>\n\n' }}
        {%- endif %}
        {%- if message['content'] %}
{{- message['content'].strip() }}
        {%- endif %}
{{- eos_token + '\n\n' }}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
{{- '### Response:\n' }}
{%- endif %}"""

# Apply template to tokenizer
tokenizer.chat_template = SHEIKH_CHAT_TEMPLATE
print("✅ Sheikh-Max chat template applied!")

# Test the template
test_messages = [
    {"role": "user", "content": "Write a hello world program"},
    {"role": "assistant", "thinking": "Simple task. I'll use Python's print function.", "content": "```python\nprint('Hello, World!')\n```"}
]

formatted = tokenizer.apply_chat_template(test_messages, tokenize=False)
print("\n📋 Template test output:")
print(formatted)

---
## 6️⃣ Prepare Training Dataset

Create a dataset with interleaved thinking examples.

In [ ]:
from datasets import Dataset

# Training data with interleaved thinking
training_data = [
    {
        "messages": [
            {"role": "system", "content": "You are Sheikh-Max, an AI that thinks step-by-step before coding."},
            {"role": "user", "content": "Write a Python function to check if a number is prime."},
            {
                "role": "assistant",
                "thinking": "A prime number is only divisible by 1 and itself. I need to:\n1. Handle edge cases (n < 2)\n2. Check divisibility from 2 to sqrt(n)\n3. Return True if no divisors found",
                "content": "```python\ndef is_prime(n: int) -> bool:\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n```"
            }
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain recursion with a simple example."},
            {
                "role": "assistant",
                "thinking": "Recursion is when a function calls itself. I should use a classic example like factorial that's easy to understand. Need to explain base case and recursive case.",
                "content": "Recursion is when a function calls itself to solve smaller versions of the same problem.\n\n**Example: Factorial**\n```python\ndef factorial(n):\n    # Base case\n    if n <= 1:\n        return 1\n    # Recursive case\n    return n * factorial(n - 1)\n```\n\nHow it works for `factorial(4)`:\n- factorial(4) = 4 × factorial(3)\n- factorial(3) = 3 × factorial(2)\n- factorial(2) = 2 × factorial(1)\n- factorial(1) = 1 (base case)\n- Result: 4 × 3 × 2 × 1 = 24"
            }
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Write a function to reverse a string without using built-in reverse."},
            {
                "role": "assistant",
                "thinking": "Multiple approaches possible:\n1. Two-pointer swap (in-place for list)\n2. Build new string from end\n3. Recursion\nI'll use the simple iteration approach for clarity.",
                "content": "```python\ndef reverse_string(s: str) -> str:\n    result = ''\n    for char in s:\n        result = char + result\n    return result\n\n# Alternative using two pointers\ndef reverse_string_v2(s: str) -> str:\n    chars = list(s)\n    left, right = 0, len(chars) - 1\n    while left < right:\n        chars[left], chars[right] = chars[right], chars[left]\n        left += 1\n        right -= 1\n    return ''.join(chars)\n```"
            }
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Implement binary search."},
            {
                "role": "assistant",
                "thinking": "Binary search requires a sorted array. Algorithm:\n1. Set left and right pointers\n2. Find middle element\n3. Compare with target\n4. Narrow search space by half\n5. Time complexity: O(log n)",
                "content": "```python\ndef binary_search(arr: list, target: int) -> int:\n    left, right = 0, len(arr) - 1\n    \n    while left <= right:\n        mid = (left + right) // 2\n        \n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    \n    return -1  # Not found\n\n# Example\narr = [1, 3, 5, 7, 9, 11, 13]\nprint(binary_search(arr, 7))  # Output: 3\n```"
            }
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Create a simple linked list class in Python."},
            {
                "role": "assistant",
                "thinking": "Need two classes:\n1. Node class to hold data and next pointer\n2. LinkedList class with methods: append, prepend, delete, display\nI'll implement the essential operations.",
                "content": "```python\nclass Node:\n    def __init__(self, data):\n        self.data = data\n        self.next = None\n\nclass LinkedList:\n    def __init__(self):\n        self.head = None\n    \n    def append(self, data):\n        new_node = Node(data)\n        if not self.head:\n            self.head = new_node\n            return\n        current = self.head\n        while current.next:\n            current = current.next\n        current.next = new_node\n    \n    def prepend(self, data):\n        new_node = Node(data)\n        new_node.next = self.head\n        self.head = new_node\n    \n    def display(self):\n        elements = []\n        current = self.head\n        while current:\n            elements.append(current.data)\n            current = current.next\n        return ' -> '.join(map(str, elements))\n\n# Usage\nll = LinkedList()\nll.append(1)\nll.append(2)\nll.prepend(0)\nprint(ll.display())  # 0 -> 1 -> 2\n```"
            }
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Write a function to find the two numbers in an array that add up to a target."},
            {
                "role": "assistant",
                "thinking": "This is the classic Two Sum problem. Options:\n1. Brute force O(n²) - check all pairs\n2. Hash map O(n) - store complements\nI'll use the hash map approach for efficiency.",
                "content": "```python\ndef two_sum(nums: list, target: int) -> list:\n    # Hash map to store number -> index\n    seen = {}\n    \n    for i, num in enumerate(nums):\n        complement = target - num\n        \n        if complement in seen:\n            return [seen[complement], i]\n        \n        seen[num] = i\n    \n    return []  # No solution found\n\n# Example\nnums = [2, 7, 11, 15]\ntarget = 9\nprint(two_sum(nums, target))  # [0, 1] because 2 + 7 = 9\n```"
            }
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do I read a JSON file in Python?"},
            {
                "role": "assistant",
                "thinking": "Simple file I/O task. Need to:\n1. Import json module\n2. Open file with context manager\n3. Use json.load() for file or json.loads() for string\nI'll show both reading and writing.",
                "content": "```python\nimport json\n\n# Reading JSON from file\nwith open('data.json', 'r') as f:\n    data = json.load(f)\n    print(data)\n\n# Writing JSON to file\nmy_data = {'name': 'Sheikh-Max', 'version': 1.0}\nwith open('output.json', 'w') as f:\n    json.dump(my_data, f, indent=2)\n\n# Parse JSON string\njson_string = '{\"key\": \"value\"}'\nparsed = json.loads(json_string)\nprint(parsed['key'])  # value\n```"
            }
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Write a decorator that measures function execution time."},
            {
                "role": "assistant",
                "thinking": "Decorators wrap functions. I need to:\n1. Import time module\n2. Create wrapper function\n3. Record start/end time\n4. Use functools.wraps to preserve metadata",
                "content": "```python\nimport time\nfrom functools import wraps\n\ndef timer(func):\n    @wraps(func)\n    def wrapper(*args, **kwargs):\n        start = time.perf_counter()\n        result = func(*args, **kwargs)\n        end = time.perf_counter()\n        print(f'{func.__name__} took {end - start:.4f} seconds')\n        return result\n    return wrapper\n\n# Usage\n@timer\ndef slow_function():\n    time.sleep(1)\n    return 'Done!'\n\nresult = slow_function()  # Prints: slow_function took 1.00xx seconds\n```"
            }
        ]
    }
]

# Create dataset
dataset = Dataset.from_list(training_data)
print(f"✅ Dataset created with {len(dataset)} examples")

# Show a sample
print("\n📋 Sample formatted conversation:")
sample = tokenizer.apply_chat_template(dataset[0]['messages'], tokenize=False)
print(sample[:500] + "...")

---
## 7️⃣ Configure QLoRA

Set up Parameter-Efficient Fine-Tuning with LoRA adapters.

In [ ]:
# QLoRA Configuration
LORA_R = 16           # LoRA rank
LORA_ALPHA = 32       # LoRA alpha (scaling factor)
LORA_DROPOUT = 0.05   # Dropout for regularization

# Target modules for Qwen architecture
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
    "gate_proj", "up_proj", "down_proj",      # MLP
]

print("🔧 Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory optimization
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied!")
print(f"   Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

---
## 8️⃣ Configure Training

Set up training arguments optimized for T4 GPU.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Hugging Face Hub settings
HF_USERNAME = os.getenv("HF_USERNAME", "OsamaBinLikhon")
HUB_MODEL_ID = f"{HF_USERNAME}/sheikh-max"

# Training configuration (optimized for T4 15GB VRAM)
training_args = TrainingArguments(
    output_dir="./results",
    
    # Batch size settings
    per_device_train_batch_size=2,      # Small batch for T4
    gradient_accumulation_steps=4,       # Effective batch = 2 * 4 = 8
    
    # Training duration
    num_train_epochs=3,                  # Or use max_steps
    # max_steps=500,
    
    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    
    # Optimizer
    optim="adamw_8bit",                  # 8-bit AdamW for memory
    
    # Precision
    fp16=True,                           # T4 doesn't support bf16
    
    # Memory optimization
    gradient_checkpointing=True,
    
    # Logging
    logging_steps=10,
    report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
    
    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    
    # Hub settings
    push_to_hub=bool(os.environ.get("HF_TOKEN")),
    hub_model_id=HUB_MODEL_ID,
    hub_token=os.environ.get("HF_TOKEN"),
)

print("✅ Training arguments configured")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Push to Hub: {training_args.push_to_hub}")

---
## 9️⃣ Initialize Trainer

In [ ]:
# Formatting function for SFTTrainer
def formatting_func(examples):
    """Format examples using the chat template."""
    texts = []
    for messages in examples['messages']:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return texts

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    formatting_func=formatting_func,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,  # Don't pack multiple examples
)

print("✅ Trainer initialized!")
print(f"   Dataset size: {len(dataset)}")
print(f"   Max sequence length: {MAX_SEQ_LENGTH}")

---
## 🔟 Start Training!

In [ ]:
print("🚀 Starting fine-tuning...")
print("="*50)

# Train!
trainer.train()

print("="*50)
print("✅ Training complete!")

---
## 1️⃣1️⃣ Test the Model

Verify that the model generates `<think>` tags.

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def generate_response(prompt, max_new_tokens=512):
    """Generate a response from Sheikh-Max."""
    messages = [
        {"role": "system", "content": "You are Sheikh-Max, an AI that thinks step-by-step before coding."},
        {"role": "user", "content": prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=False)
    return response

# Test prompts
test_prompts = [
    "Write a Python function to calculate the Fibonacci sequence.",
    "How do I sort a list of dictionaries by a specific key?",
    "Implement a stack data structure."
]

print("🧪 Testing Sheikh-Max...\n")
for i, prompt in enumerate(test_prompts, 1):
    print(f"{'='*60}")
    print(f"Test {i}: {prompt}")
    print(f"{'='*60}")
    
    response = generate_response(prompt)
    print(response)
    
    # Check for <think> tags
    if "<think>" in response and "</think>" in response:
        print("\n✅ Interleaved thinking detected!")
    else:
        print("\n⚠️ No <think> tags found in response.")
    print()

---
## 1️⃣2️⃣ Save Model

In [ ]:
# Save locally
print("💾 Saving model locally...")
trainer.save_model("./results/sheikh-max-final")
tokenizer.save_pretrained("./results/sheikh-max-final")
print("✅ Model saved to ./results/sheikh-max-final")

# Push to Hub if token available
if os.environ.get("HF_TOKEN"):
    print(f"\n📤 Pushing to Hugging Face Hub: {HUB_MODEL_ID}")
    trainer.push_to_hub()
    print(f"✅ Model pushed to https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("\n⚠️ HF_TOKEN not set. Model saved locally only.")

---
## 1️⃣3️⃣ Export to GGUF (Optional)

Export the model to GGUF format for use with Ollama or LM Studio.

In [ ]:
# Uncomment to export to GGUF
# This requires additional disk space

# print("📦 Exporting to GGUF format...")
# model.save_pretrained_gguf(
#     "./results/sheikh-max-gguf",
#     tokenizer,
#     quantization_method="q4_k_m"  # Good balance of size/quality
# )
# print("✅ GGUF export complete!")

# # Push GGUF to Hub
# if os.environ.get("HF_TOKEN"):
#     model.push_to_hub_gguf(
#         f"{HUB_MODEL_ID}-GGUF",
#         tokenizer,
#         quantization_method="q4_k_m",
#         token=os.environ.get("HF_TOKEN")
#     )
#     print(f"✅ GGUF pushed to {HUB_MODEL_ID}-GGUF")

---
## 🎉 Done!

Sheikh-Max has been fine-tuned with interleaved thinking capabilities!

### Next Steps:
1. **Test more prompts** to verify `<think>` tag generation
2. **Train on more data** for better results (try Bespoke-Stratos-17k or OpenThoughts)
3. **Deploy** using the inference script or Gradio demo

### Resources:
- [Model on HuggingFace](https://huggingface.co/OsamaBinLikhon/sheikh-max)
- [GitHub Repository](https://github.com/osamabinlikhon/sheikh-max)
- [Unsloth Documentation](https://github.com/unslothai/unsloth)